In [ ]:
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from tensorflow.keras.models import clone_model
from tensorflow.keras.optimizers import Adam
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
import os
import pandas as pd
from tensorflow.keras.models import load_model
import seaborn as sns
from sklearn.metrics import mean_absolute_error
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

In [ ]:
# Cargar conjuntos estratificados
X_train_cls = np.load('data/X_train_class.npy')
X_test_cls = np.load('data/X_test_class.npy')

### Generar imágenes ruidosas

In [ ]:
import numpy as np

# Lista de factores de ruido a probar
noise_factors = [0.2, 0.39, 0.6]

# Cargar imágenes limpias
X_train_clean = np.load("data/X_train_class.npy")
X_test_clean = np.load("data/X_test_class.npy")

# Generar y guardar datasets ruidosos para cada factor
for nf in noise_factors:
    print(f"Generando imágenes con ruido gaussiano: factor = {nf}")
    
    # Generar ruido gaussiano
    X_train_noisy = X_train_clean + nf * np.random.normal(loc=0.0, scale=1.0, size=X_train_clean.shape)
    X_test_noisy = X_test_clean + nf * np.random.normal(loc=0.0, scale=1.0, size=X_test_clean.shape)

    # Limitar valores entre 0 y 1
    X_train_noisy = np.clip(X_train_noisy, 0., 1.)
    X_test_noisy = np.clip(X_test_noisy, 0., 1.)

    # Convertir nf a string sin punto decimal para el nombre de archivo (ej: 0.39 -> "039")
    nf_str = str(nf).replace(".", "")
    
    # Guardar archivos
    np.save(f"data/train_gauss_{nf_str}.npy", X_train_noisy)
    np.save(f"data/test_gauss_{nf_str}.npy", X_test_noisy)

print("✅ Listo: todos los datasets ruidosos fueron generados y guardados.")


In [ ]:

X_prueba = X_train_clean[:2].copy()

rng = np.random.default_rng(42)
ruido = rng.normal(loc=0.0, scale=1.0, size=X_prueba.shape)
X_gauss_prueba = np.clip(X_prueba + 0.39 * ruido, 0.0, 1.0)

print("Dimensiones:", X_gauss_prueba.shape)
print("Rango:", X_gauss_prueba.min(), X_gauss_prueba.max())
print("Píxeles modificados:", np.count_nonzero(X_prueba != X_gauss_prueba))
  

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Lista de factores de ruido que generaste
noise_factors = [0.2, 0.39, 0.6]

# Cargar imágenes limpias
X_clean = np.load("data/X_train_class.npy")

# Número de imágenes a mostrar
n = 10
indices = np.random.choice(len(X_clean), n, replace=False)

# Mostrar comparación para cada nivel de ruido
for nf in noise_factors:
    nf_str = str(nf).replace(".", "")  # para el nombre de archivo
    X_noisy = np.load(f"data/train_gauss_{nf_str}.npy")

    # Asegurar consistencia
    assert len(X_clean) == len(X_noisy)

    # Crear figura
    plt.figure(figsize=(18, 4))
    for i, idx in enumerate(indices):
        # Imágenes limpias
        ax = plt.subplot(2, n, i + 1)
        plt.imshow(X_clean[idx].squeeze(), cmap="gray")
        if i == 0:
            ax.set_ylabel("Limpia", fontsize=12)
        plt.axis("off")

        # Imágenes ruidosas
        ax = plt.subplot(2, n, i + 1 + n)
        plt.imshow(X_noisy[idx].squeeze(), cmap="gray")
        if i == 0:
            ax.set_ylabel(f"Ruido {nf}", fontsize=12)
        plt.axis("off")

    plt.suptitle(f"Comparación de imágenes: Limpias vs. Ruidosas (factor={nf})", fontsize=14)
    plt.tight_layout()
    plt.show()



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Factores de ruido
noise_factors = [0.2, 0.39, 0.6]

# Cargar imágenes limpias
X_clean = np.load("data/X_train_class.npy")

# Número de columnas
n_cols = 7
indices = np.random.choice(len(X_clean), n_cols, replace=False)

# Crear figura
fig, axes = plt.subplots(4, n_cols, figsize=(2*n_cols, 8))

# --- Fila 1: imágenes originales ---
for j, idx in enumerate(indices):
    ax = axes[0, j]
    ax.imshow(X_clean[idx].squeeze(), cmap="gray")
    ax.axis("off")
# Etiqueta de la fila
axes[0, 0].set_ylabel("Original", fontsize=12, labelpad=20, rotation=0, va="center")

# --- Filas siguientes ---
row_titles = ["Ruido 0.20", "Ruido 0.39", "Ruido 0.60"]
for i, nf in enumerate(noise_factors, start=1):
    nf_str = str(nf).replace(".", "")
    X_noisy = np.load(f"data/train_gauss_{nf_str}.npy")

    for j, idx in enumerate(indices):
        ax = axes[i, j]
        ax.imshow(X_noisy[idx].squeeze(), cmap="gray")
        ax.axis("off")

    # Etiqueta de la fila
    axes[i, 0].set_ylabel(row_titles[i-1], fontsize=12, labelpad=20,
                          rotation=0, va="center")

# Título general
plt.suptitle("Comparación de imágenes con diferentes niveles de ruido gaussiano", fontsize=16)
plt.tight_layout()
plt.show()


### Modelo PEQUEÑO para ruido gaussiano

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam

# Arquitectura reducida del autoencoder
input_img = layers.Input(shape=(64, 128, 1)) 

# --- Encoder ---
x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(input_img)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D((2, 2), padding='same')(x)

x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D((2, 2), padding='same')(x)

# --- Bottleneck ---
x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)

# --- Decoder ---
x = layers.UpSampling2D((2, 2))(x)
x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = layers.BatchNormalization()(x)

x = layers.UpSampling2D((2, 2))(x)
x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
x = layers.BatchNormalization()(x)

# --- Salida ---
output_img = layers.Conv2D(1, (3, 3), activation='sigmoid', padding='same')(x)

# Modelo
autoencoder_small = models.Model(input_img, output_img)
autoencoder_small.compile(optimizer=Adam(learning_rate=1e-3),
                          loss='mean_squared_error',
                          metrics=['mae'])

In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tensorflow.keras.models import clone_model
from tensorflow.keras.optimizers import Adam

# Lista de factores de ruido
noise_factors = [0.2, 0.39, 0.6]

# Directorio de datos y modelos
ruido_dir = "data"
model_dir = "modelos_autoencoder_small"
os.makedirs(model_dir, exist_ok=True)

# Datos limpios
X_train_clean = np.load(os.path.join(ruido_dir, "X_train_class.npy"))
X_test_clean = np.load(os.path.join(ruido_dir, "X_test_class.npy"))

# Diccionario para guardar métricas
mae_denoise = {}

# Entrenar con cada nivel de ruido
for nf in noise_factors:
    nf_str = str(nf).replace(".", "")  # para nombres de archivo
    print(f"\nEntrenando con ruido gaussiano: factor = {nf}")

    # Cargar datos ruidosos
    train_name = f"train_gauss_{nf_str}.npy"
    test_name = f"test_gauss_{nf_str}.npy"
    X_train_noisy = np.load(os.path.join(ruido_dir, train_name))
    X_test_noisy = np.load(os.path.join(ruido_dir, test_name))

    # Clonar y compilar modelo
    modelo_denoise = clone_model(autoencoder_small)  # tu autoencoder reducido
    modelo_denoise.compile(optimizer=Adam(learning_rate=1e-3),
                           loss='mean_squared_error',
                           metrics=['mae'])

    # Entrenar
    historial = modelo_denoise.fit(
        X_train_noisy, X_train_clean,
        epochs=50,
        batch_size=32,
        shuffle=True,
        validation_data=(X_test_noisy, X_test_clean),
        verbose=0
    )

    # Guardar modelo entrenado
    model_name = f"autoencoder_small_gauss_{nf_str}.keras"
    modelo_denoise.save(os.path.join(model_dir, model_name))

    # Guardar MAE de validación
    mae_val = historial.history['val_mae'][-1]
    mae_denoise[nf] = mae_val
    print(f"MAE de validación (factor={nf}): {mae_val:.4f}")

# Convertir diccionario a DataFrame
df_mae = pd.DataFrame.from_dict(mae_denoise, orient='index', columns=['MAE'])

# Dibujar heatmap
plt.figure(figsize=(6, 4))
sns.heatmap(df_mae.T, annot=True, fmt=".4f", cmap="YlGnBu", cbar_kws={'label': 'MAE'})
plt.title("Heatmap del MAE por factor de ruido gaussiano")
plt.xlabel("Factor de ruido")
plt.ylabel("Métrica")
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model

# Factores de ruido y directorio de modelos
noise_factors = [0.2, 0.39, 0.6]
ruido_dir = "data"
model_dir = "modelos_autoencoder_small"

# Cargar imágenes limpias (usamos test para visualización)
X_clean = np.load(f"{ruido_dir}/X_test_class.npy")

# Selección de ejemplos aleatorios
n = 5
indices = np.random.choice(len(X_clean), n, replace=False)

# Comparación por cada factor de ruido
for nf in noise_factors:
    nf_str = str(nf).replace(".", "")
    print(f"\nMostrando comparaciones para factor de ruido = {nf}")

    # Cargar imágenes ruidosas
    X_noisy = np.load(f"{ruido_dir}/test_gauss_{nf_str}.npy")

    # Cargar modelo entrenado correspondiente
    model_path = f"{model_dir}/autoencoder_small_gauss_{nf_str}.keras"
    modelo = load_model(model_path)

    # Generar predicciones
    X_pred = modelo.predict(X_noisy[indices])
    # Binarizar predicciones con umbral 0.5
    X_pred_bin = (X_pred > 0.5).astype(np.float32)

    # Visualización
    plt.figure(figsize=(16, 8))
    for i, idx in enumerate(indices):
        # Original
        ax = plt.subplot(4, n, i + 1)
        plt.imshow(X_clean[idx].squeeze(), cmap="gray")
        if i == 0: ax.set_ylabel("Original", fontsize=12)
        plt.axis("off")

        # Ruidosa
        ax = plt.subplot(4, n, i + 1 + n)
        plt.imshow(X_noisy[idx].squeeze(), cmap="gray")
        if i == 0: ax.set_ylabel("Ruidosa", fontsize=12)
        plt.axis("off")

        # Predicha
        ax = plt.subplot(4, n, i + 1 + 2*n)
        plt.imshow(X_pred[i].squeeze(), cmap="gray")
        if i == 0: ax.set_ylabel("Predicha", fontsize=12)
        plt.axis("off")

        # Predicha binarizada
        ax = plt.subplot(4, n, i + 1 + 3*n)
        plt.imshow(X_pred_bin[i].squeeze(), cmap="gray")
        if i == 0: ax.set_ylabel("Binaria", fontsize=12)
        plt.axis("off")

    plt.suptitle(f"Comparación por factor de ruido {nf}", fontsize=14)
    plt.tight_layout()
    plt.show()


In [ ]:
import os
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.losses import MeanSquaredError
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

# Factores de ruido
noise_factors = [0.2, 0.39, 0.6]
ruido_dir = "data"
model_dir = "modelos_autoencoder_small"

# Datos limpios (test)
X_test_clean = np.load(os.path.join(ruido_dir, "X_test_class.npy"))

# Función de pérdida
mse_fn = MeanSquaredError()

# Diccionario para guardar resultados
results = {}

# Calcular métricas para cada modelo
for nf in noise_factors:
    nf_str = str(nf).replace(".", "")
    print(f"\nEvaluando modelo con factor de ruido = {nf}")

    # Cargar datos ruidosos de test
    X_test_noisy = np.load(os.path.join(ruido_dir, f"test_gauss_{nf_str}.npy"))

    # Cargar modelo entrenado
    model_path = os.path.join(model_dir, f"autoencoder_small_gauss_{nf_str}.keras")
    modelo = load_model(model_path)

    # Predicciones
    preds = modelo.predict(X_test_noisy)
    preds_bin = (preds > 0.5).astype(np.float32)

    # MSE
    loss_mse = mse_fn(X_test_clean, preds_bin).numpy()

    # MAE
    mae = np.mean(np.abs(X_test_clean - preds_bin))

    # PSNR
    psnr_vals = [psnr(gt.squeeze(), pr.squeeze(), data_range=1.0) 
                 for gt, pr in zip(X_test_clean, preds_bin)]
    psnr_avg = np.mean(psnr_vals)

    # SSIM
    ssim_vals = [ssim(gt.squeeze(), pr.squeeze(), data_range=1.0) 
                 for gt, pr in zip(X_test_clean, preds_bin)]
    ssim_avg = np.mean(ssim_vals)

    # Guardar resultados
    results[nf] = {
        "MSE": loss_mse,
        "MAE": mae,
        "PSNR": psnr_avg,
        "SSIM": ssim_avg
    }

    # Mostrar resultados en consola
    print(f"Loss (MSE): {loss_mse:.4f}")
    print(f"MAE:        {mae:.4f}")
    print(f"PSNR (dB):  {psnr_avg:.2f}")
    print(f"SSIM:       {ssim_avg:.4f}")

# ---- Opcional: convertir a tabla ----
import pandas as pd
df_results = pd.DataFrame(results).T
print("\nResumen de métricas por factor de ruido:")
print(df_results)
